In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import dask.dataframe as dd
import numpy as np
import datashader as ds
import datashader.transfer_functions as tf
import geopandas as gpd
import pandas as pd
from shapely.ops import polygonize,unary_union
from matplotlib.colors import LinearSegmentedColormap, LightSource
from matplotlib import pyplot as plt
from matplotlib.colors import Normalize, LightSource
from shapely import Point,LineString
import pickle
from pyproj import Transformer
from PIL import Image

from aquacrop_slovenia import config

In [ ]:
colors = [
    (0, (0.197428, 0.430237, 0.315623)),
    (0.01, (0.374411, 0.598965, 0.478371)), # zelena
    (0.03, (0.568697, 0.680937, 0.554058)), # svetlo zelena
    (0.15, (0.842646, 0.779653, 0.579218)), # rumena
    (0.28, (0.797949, 0.577794, 0.352955)), # oranžna
    (0.38, (0.514270, 0.401949, 0.377880)), # rjava
    (0.6, (0.730542, 0.702137, 0.685094)), # skoraj bela
    (0.68, (1, 1, 1)), # bela
    (1, (1, 1, 1)) # bela
]

positions, color_vals = zip(*colors)

terrain_cmap = LinearSegmentedColormap.from_list("custom_terrain", list(zip(positions, color_vals)))

In [ ]:
terrain_cmap

In [ ]:
def hillshade_agg(agg, azimuth=315, altitude=45):
    elev = agg.data

    az = np.radians(azimuth)
    alt = np.radians(altitude)

    dx, dy = np.gradient(elev)
    slope = np.pi/2 - np.arctan(np.sqrt(dx*dx + dy*dy))
    aspect = np.arctan2(-dx, dy)

    shaded = (
        np.sin(alt) * np.sin(slope) +
        np.cos(alt) * np.cos(slope) * np.cos(az - aspect)
    )

    shaded = 255 * (shaded + 1) / 2
    return shaded

def removeseaborder(x):
    coords = list(x.coords)
    newcoords = []
    for i in range(len(coords)):
        if i > 4:
            newcoords.append(Point(coords[i][0], coords[i][1]))
    return LineString(newcoords)

def removeseaborder2(x):
    coords = list(x.coords)
    imax = (len(coords)-3)
    newcoords = []
    for i in range(len(coords)):
        if i < imax:
            newcoords.append(Point(coords[i][0], coords[i][1]))
    return LineString(newcoords)

In [ ]:
ddf = dd.read_csv(
    str(config.EXTERNAL_MAPDATA_DIR / "DMV0250/*.XYZ"),
    sep=r'\s+',
    names=["x", "y", "z"],
    dtype={"x": "float64", "y": "float64", "z": "float64"}
)

In [ ]:
xmin, xmax = ddf.x.min().compute(), ddf.x.max().compute()
ymin, ymax = ddf.y.min().compute(), ddf.y.max().compute()

print((xmax-xmin)/(ymax-ymin))

In [ ]:
gdf_border = gpd.read_file(config.EXTERNAL_MAPDATA_DIR / "DRZAVNA_MEJA/EDM-mejna_crta.shp")

merged_border = unary_union(gdf_border.geometry)
border_polygons = unary_union(polygonize(merged_border))

def points_in_polygon(partition_df, polygon):
    gdf_points = gpd.GeoSeries.from_xy(partition_df.x, partition_df.y, crs=gdf_border.crs)
    mask = gdf_points.within(polygon)
    return partition_df[mask]

ddf_nosea = ddf[ddf.z != 0]
demdf_insideborder = ddf_nosea.map_partitions(points_in_polygon, border_polygons)

In [ ]:
canvas = ds.Canvas(
    plot_width=6000,
    plot_height=4000,
    x_range=(xmin, xmax),
    y_range=(ymin, ymax)
)

In [ ]:
# takes more than 10 min !!! so we pickle it for later
# If the pickle file exsists, there is no need to run this cell, load the variable from file in the next cell
agg_clipped = canvas.points(demdf_insideborder, 'x', 'y', agg=ds.mean('z'))

In [ ]:
with open(config.INTERIM_MAPDATA_DIR / "agg_clipped.pkl", "wb") as f:
    pickle.dump(agg_clipped, f)

In [ ]:
# load aggregated data
with open(config.INTERIM_MAPDATA_DIR / "agg_clipped.pkl", "rb") as f:
    agg_clipped = pickle.load(f)

In [ ]:
gdf_border.loc[0, "geometry"] = removeseaborder(gdf_border.geometry[0])
gdf_border.loc[5, "geometry"] = removeseaborder2(gdf_border.geometry[5])

parts = []
for geom in gdf_border.geometry:
    if geom.geom_type == "LineString":
        xs, ys = geom.xy
        parts.append(pd.DataFrame({"x": xs, "y": ys}))

with_separators = []
for df_part in parts:
    with_separators.append(df_part)
    with_separators.append(pd.DataFrame({"x": [np.nan], "y": [np.nan]}))

border_df = pd.concat(with_separators, ignore_index=True)
border_agg = canvas.line(border_df, "x", "y", agg=ds.count())
border_img = tf.shade(border_agg, cmap=["black"], how="linear")
border_img = tf.spread(border_img, px=1)

# --- Elevation + hillshade ---
elev_array = agg_clipped.data
land_mask = ~np.isnan(elev_array)
elev_filled = np.where(land_mask, elev_array, 0.0)

dx_m = (xmax - xmin) / canvas.plot_width   # real-world pixel size, for correct slope angles
dy_m = (ymax - ymin) / canvas.plot_height

ls = LightSource(azdeg=315, altdeg=45)
norm_elev = Normalize(vmin=0, vmax=np.nanmax(elev_array))

terrain_rgba = ls.shade(
    elev_filled,
    cmap=terrain_cmap,
    norm=norm_elev,
    blend_mode="soft",
    vert_exag=5,
    fraction=0.8,
    dx=dx_m,
    dy=dy_m,
)
terrain_rgba = (terrain_rgba * 255).astype(np.uint8)
terrain_rgba[~land_mask, 3] = 0  # keep outside-border pixels transparent, as before
terrain_rgba = np.flipud(terrain_rgba)

terrain_img = Image.fromarray(terrain_rgba, mode="RGBA")

In [ ]:
terrain_img

In [ ]:
gdf_water = gpd.read_file(config.EXTERNAL_MAPDATA_DIR / "hidrografija/P250V5000_pG.shp")
gdf_water_clipped = gpd.clip(gdf_water[:1330], border_polygons)
water_full_agg = canvas.polygons(gdf_water_clipped, geometry='geometry', agg=ds.count())
water_border_agg = canvas.line(gdf_water_clipped, geometry='geometry', agg=ds.count())
water_fill_img = tf.shade(water_full_agg, cmap=["lightblue"], how="eq_hist", alpha=240)
water_border_img = tf.shade(water_border_agg, cmap=["#00a6ff"], alpha=255)
water_img = tf.stack(water_fill_img, water_border_img, how="over")

In [ ]:
gdf_water_small = gpd.read_file(config.EXTERNAL_MAPDATA_DIR / "hidrografija/P250V5000_lG.shp")
gdf_water_small_clipped = gpd.clip(gdf_water_small, border_polygons)
water_small_agg = canvas.line(gdf_water_small_clipped, geometry='geometry', agg=ds.count())
water_small_img = tf.shade(water_small_agg, cmap=["#4B9BBC"], alpha=200)

In [ ]:
sea_polygon = gdf_water.geometry[1330]
gdf_sea = gpd.GeoDataFrame(pd.DataFrame({'geometry': [sea_polygon]}), crs=gdf_border.crs)
sea_full_agg = canvas.polygons(gdf_sea, geometry='geometry', agg=ds.count())
sea_border_agg = canvas.line(gdf_sea, geometry='geometry', agg=ds.count())
sea_fill_img = tf.shade(sea_full_agg, cmap=["lightblue"], how="eq_hist", alpha=240)
sea_border_img = tf.shade(sea_border_agg, cmap=["#00a6ff"], alpha=255)
sea_img = tf.stack(sea_fill_img, sea_border_img, how="over")

In [ ]:
overlays_img = tf.stack(sea_img, water_small_img, water_img, border_img, how="over").to_pil().convert("RGBA")

final_water_sea = Image.alpha_composite(terrain_img, overlays_img)

background = Image.new("RGBA", final_water_sea.size, "white")
final_water_sea = Image.alpha_composite(background, final_water_sea)

dsimage = final_water_sea.convert("RGB")
#dsimage.save("output/final_water_sea_dem_4000.png")

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
im = ax.imshow(dsimage)
ax.set_axis_off()
plt.tight_layout()
plt.savefig("figures/final_water_sea_dem_mpl_small.png", dpi=600)

In [ ]:
norm = Normalize(vmin=0, vmax=agg_clipped.max())

fig, ax = plt.subplots(figsize=(7,4))

im = ax.imshow(dsimage, norm=norm, cmap=terrain_cmap)

markers = [
    {"name": "Jablje polje KIS", "lat": 46.143634, "lon": 14.557040, "kind": "field"},
    {"name": "Letališče JP LJ ARSO", "lat": 46.2175, "lon": 14.473056, "kind": "station"},
    {"name": "Rakičan polje KIS", "lat": 46.653648, "lon": 16.189925, "kind": "field"},
    {"name": "Murska Sobota ARSO", "lat": 46.652222, "lon": 16.191389, "kind": "station"},
]

kind_style = {
    "field": dict(color="red", label="KIS field"),
    "station": dict(color="blue", label="ARSO station"),
}

transformer = Transformer.from_crs("EPSG:4326", "EPSG:3794", always_xy=True)

seen_kinds = set()
for m in markers:
    x_real, y_real = transformer.transform(m["lon"], m["lat"])  # (lon, lat) order!

    x_px = (x_real - xmin) / (xmax - xmin) * canvas.plot_width
    y_px = (ymax - y_real) / (ymax - ymin) * canvas.plot_height  # flip: image y grows downward

    style = kind_style[m["kind"]]
    label = style["label"] if m["kind"] not in seen_kinds else None
    seen_kinds.add(m["kind"])

    ax.scatter(x_px, y_px, marker='.', color=style["color"], s=20, label=label)

ax.legend(loc="center right", frameon=False, fontsize=9)

cbar = fig.colorbar(im, ax=ax, extend="max")
cbar.ax.set_ylim(0, 2000)
cbar.set_label('Elevation above sea level [m]')
ax.set_axis_off()

# --- Scale bar ---
# The image is in pixel space; xmin/xmax are the real-world (EPSG:3794, meters)
# extent used to build `canvas`, so we can convert a physical length to pixels.
def add_scale_bar(ax, canvas, xmin, xmax, tick_km=(0, 10, 20, 50), right_margin=0.05, bottom_margin=0.02, color="black", fontsize=11):
    length_km = max(tick_km)
    meters_per_pixel = (xmax - xmin) / canvas.plot_width
    bar_length_px = (length_km * 1000) / meters_per_pixel

    x1 = canvas.plot_width * (1 - right_margin)  # right end of the bar
    x0 = x1 - bar_length_px                      # left end of the bar
    y0 = canvas.plot_height * (1 - bottom_margin)

    ax.plot([x0, x1], [y0, y0], color=color, lw=1.5, solid_capstyle="butt")

    tick_h = canvas.plot_height * 0.01
    for km in tick_km:
        xi = x0 + (km / length_km) * bar_length_px
        ax.plot([xi, xi], [y0 - tick_h, y0], color=color, lw=1.2)  # ticks above the line only
        if km not in [30, 40]:
            ax.text(xi, y0 - tick_h - 5, f"{km:.0f}", ha="center", va="bottom", fontsize=fontsize, color=color, fontfamily="sans-serif")

    ax.text(x1 + 65, y0, "km", ha="left", va="center", fontsize=fontsize, color=color, fontfamily="sans-serif")

    return x0, x1, y0  # extent of the bar, for aligning other elements

x0_scale, x1_scale, y0_scale = add_scale_bar(ax, canvas, xmin, xmax, tick_km=(0, 10, 20, 30, 40, 50), fontsize=9)

# --- North arrow ---
# Classic cartographic "half-arrow" symbol: a kite shape split down the middle,
# left half white / right half black, both with a thin black outline.
# EPSG:3794 is a conformal transverse Mercator over a small extent, so "up" in
# pixel space is a very close approximation of true north here.
from matplotlib.patches import Polygon

def add_north_arrow(ax, canvas, x, y_base, gap_frac=0.1, length_frac=0.09, width_frac=0.035, color="black"):
    y_bottom = y_base - canvas.plot_height * gap_frac  # base of the arrow, just above the scale bar
    length = canvas.plot_height * length_frac
    width = canvas.plot_width * width_frac
    y_top = y_bottom - length                           # tip (points north)
    y_mid = (y_top + y_bottom) / 2

    # Fills carry no stroke of their own: two separately-stroked halves each drawing
    # their own edge up to the tip don't rasterize to exactly the same pixel, which
    # made the tip look split/misaligned. A single outline polygon on top traces the
    # whole kite so the tip and outer edges are drawn exactly once.
    left_fill = Polygon(
        [(x, y_top), (x - width / 2, y_bottom), (x, y_mid)],
        closed=True, facecolor="white", edgecolor="none", zorder=5,
    )
    right_fill = Polygon(
        [(x, y_top), (x + width / 2, y_bottom), (x, y_mid)],
        closed=True, facecolor=color, edgecolor="none", zorder=5,
    )
    outline = Polygon(
        [(x, y_top), (x + width / 2, y_bottom), (x, y_mid), (x - width / 2, y_bottom)],
        closed=True, facecolor="none", edgecolor=color, linewidth=1.2, zorder=6,
    )
    ax.add_patch(left_fill)
    ax.add_patch(right_fill)
    ax.add_patch(outline)

    ax.text(x, y_top - canvas.plot_height * 0.015, "N",
            ha="center", va="bottom", fontsize=11, fontweight="bold", color=color, fontfamily="sans-serif")

add_north_arrow(ax, canvas, x=x1_scale + 150, y_base=y0_scale)  # right above the scale bar, near the bottom of the map

plt.tight_layout()
plt.savefig("figures/map_stations_fields.png", dpi=500)